In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
# These imports are required for IterativeImputer
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer

In [3]:
# Your helper function (kept as is, but used safely below)
def string_to_bool(s: str) -> bool:
    s_lower = s.lower()
    if s_lower in ['true', 'yes', '1', 't', 'y']:
        return True
    elif s_lower in ['false', 'no', '0', 'f', 'n']:
        return False
    else:
        raise ValueError(f"Cannot convert '{s}' to a boolean.")


def remove_outliers(df):
    numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
    for col in numerical_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    return df


    

class ImputationHandler:
    def __init__(self, df):
        self.df = df
        self.numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
        self.categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

    def get_missing_report(self):
        """Returns a Series of columns with missing values and their counts."""
        nulls = self.df.isna().sum()
        return nulls[nulls > 0]

    def _get_strategy_map(self):
        """Defines the menu options."""
        return { 
            'numerical':{
                '0': 'Drop Rows',
                '1': 'Mean',
                '2': 'Median',
                '3': 'Mode',
                '4': 'Manual Constant',
                '5': 'Forward Fill',
                '6': 'Backward Fill',
                '7': 'Interpolation (Linear)',
                '8': 'KNN Imputation (Multivariate)',
                '9': 'Iterative Imputer (Multivariate)'
            },
            'categorical':{
                '0': 'Drop Rows',
                '1': 'Most Frequent (Mode)',
                '2': 'Placeholder (e.g., "Missing")',
                '3': 'Forward Fill',
                '4': 'Backward Fill'
            }
        }

    def ask_strategy(self, dtype):
        """Displays the menu and returns the user's choice."""
        strategies = self._get_strategy_map()[dtype]
        print(f"\n--- Available Strategies for {dtype.upper()} ---")
        for k, v in strategies.items():
            print(f"{k}: {v}")
        
        choice = input("Select a number: ").strip()
        return choice if choice in strategies else None

    def apply_imputation(self, cols, strategy_idx, dtype):
        """
        Applies the chosen strategy to the specified columns.
        Returns the modified dataframe.
        """
        # --- Handle Multivariate Methods (Applied to whole subset) ---
        if dtype == 'numerical' and strategy_idx in ['8', '9']:
            return self._apply_multivariate(cols, strategy_idx)

        # --- Handle Univariate Methods (Column by Column) ---
        for col in cols:
            print(f"Applying strategy {strategy_idx} to {col}...")
            
            if strategy_idx == '0': # Drop
                self.df.dropna(subset=[col], inplace=True)
            
            elif strategy_idx == '1': # Mean/Mode
                val = self.df[col].mean() if dtype == 'numerical' else self.df[col].mode()[0]
                self.df[col] = self.df[col].fillna(val)

            elif strategy_idx == '2': # Median / Constant
                if dtype == 'numerical':
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                else:
                    self.df[col] = self.df[col].fillna("Missing_Value")

            elif strategy_idx == '3' and dtype == 'numerical': # Mode for numeric
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])

            elif strategy_idx == '4': # Manual
                val = input(f"Enter value for {col}: ")
                # Convert to float if numerical
                if dtype == 'numerical':
                    val = float(val)
                self.df[col] = self.df[col].fillna(val)

            elif strategy_idx == '5' or (strategy_idx == '3' and dtype == 'categorical'): # Ffill
                self.df[col] = self.df[col].ffill()
            
            elif strategy_idx == '6' or (strategy_idx == '4' and dtype == 'categorical'): # Bfill
                self.df[col] = self.df[col].bfill()

            elif strategy_idx == '7' and dtype == 'numerical': # Interpolate
                self.df[col] = self.df[col].interpolate(method='linear')

        return self.df

    def _apply_multivariate(self, cols, strategy_idx):
        """Handles KNN and Iterative Imputer."""
        # We generally need all numerical columns for context, not just the missing ones
        context_cols = self.numerical_cols
        
        print("\nRunning Multivariate Imputation... (This uses all numerical columns for context)")
        
        if strategy_idx == '8': # KNN
            # n_neighbors is adjustable
            imputer = KNNImputer(n_neighbors=5) 
            self.df[context_cols] = imputer.fit_transform(self.df[context_cols])
            print("KNN Imputation Complete.")

        elif strategy_idx == '9': # Iterative
            imputer = IterativeImputer(max_iter=10, random_state=0)
            self.df[context_cols] = imputer.fit_transform(self.df[context_cols])
            print("Iterative Imputation Complete.")
            
        return self.df







from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

class EncodingHandler:
    def __init__(self, df, target_col=None):
        self.df = df
        self.target_col = target_col
        self.options = {
            '1': 'One Hot Encoding',
            '2': 'Label Encoding',
            '3': 'Ordinal Encoding (Auto)',
            '4': 'Target Guided Encoding (Mean Encoding)'
        }

    def ask_strategy(self):
        print("\n--- Encoding Options ---")
        for k, v in self.options.items():
            print(f"{k}: {v}")
        return input("Select encoding method (number): ").strip()

    def apply_encoding(self, col, strategy):
        """
        Applies encoding to a SINGLE column. 
        Returns the modified DataFrame.
        """
        print(f"Encoding {col} using {self.options.get(strategy, 'Unknown')}...")

        if strategy == '1': # One Hot Encoding
            # pd.get_dummies creates new cols. We join them and drop the original.
            dummies = pd.get_dummies(self.df[col], prefix=col, drop_first=True)
            # Boolean to int (True/False -> 1/0)
            dummies = dummies.astype(int)
            self.df = pd.concat([self.df, dummies], axis=1)
            self.df.drop(columns=[col], inplace=True)

        elif strategy == '2': # Label Encoding
            le = LabelEncoder()
            self.df[col] = le.fit_transform(self.df[col])

        elif strategy == '3': # Ordinal Encoding (Auto)
            # Note: Real ordinal encoding usually requires a specific list of order 
            # (e.g., Low < Medium < High). This version assigns arbitrary integers.
            oe = OrdinalEncoder()
            self.df[col] = oe.fit_transform(self.df[[col]])

        elif strategy == '4': # Target Guided
            if self.target_col is None:
                print(f"Error: Cannot perform Target Encoding on {col} without a target column.")
                return self.df
            
            # 1. Calculate mean of target for each category
            mean_labels = self.df.groupby(col)[self.target_col].mean()
            # 2. Map these means to the original column
            self.df[col] = self.df[col].map(mean_labels)

        return self.df